# MTT — Multilingual Translation Transformer · Google Colab
**Encoder:** mmBERT-small (ModernBERT + Gemma 2 tokenizer, 140 M params)  
**Decoder:** MT5-small (250 k SentencePiece vocab)  
**NER:** spaCy `xx_ent_wiki_sm` → uncertainty-weighted multi-task loss  
**Languages:** de · fr · vi · es · en (pivot pairs generated automatically)

**Colab workflow**
1. Run **§1 GPU check** — choose the right config preset below  
2. Run **§2 Install** → **§3 Drive** → **§4 Write files**  
3. Edit **§5 Config** if needed, then run **§6 → §7 → §8** to train  
4. Re-run **§8** after a disconnect to resume from the last checkpoint  
5. Use **§9 Inference** to test translations at any time  
6. Use **§10 Download** to grab the checkpoint locally  

> ⚠️ Colab free tier gives ~12 GB VRAM (T4). The default config is tuned for T4.  
> Colab Pro / Pro+ gives L4 (22 GB) or A100 (40 GB) — use the preset comments in §5.

## §1 · GPU Check

In [ ]:
import torch, subprocess, os

os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

if not torch.cuda.is_available():
    raise RuntimeError("No GPU found!  Runtime → Change runtime type → T4 GPU")

gpu = torch.cuda.get_device_name(0)
vram = torch.cuda.get_device_properties(0).total_memory / 1024**3
print(f"GPU  : {gpu}")
print(f"VRAM : {vram:.1f} GB")
print(f"CUDA : {torch.version.cuda}")
print()
if   vram >= 38: print("✅ A100 detected  — use A100 preset in §5 Config")
elif vram >= 20: print("✅ L4 detected    — use L4 preset in §5 Config")
elif vram >= 14: print("✅ T4 detected    — use default T4 preset in §5 Config")
else:            print("⚠️  Small VRAM   — reduce batch_size / accum_steps in §5")


## §2 · Install Dependencies

In [ ]:
# Takes ~2 min on first run; skip if already installed in this session
!pip install -q torch transformers datasets sentencepiece spacy
!python -m spacy download xx_ent_wiki_sm -q
print("✅ All packages installed")


## §3 · Mount Google Drive
Checkpoints are saved here so they survive Colab disconnects.  
Skip this cell if you don't want Drive persistence.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

import os
DRIVE_DIR = "/content/drive/MyDrive/mtt_checkpoints"
os.makedirs(DRIVE_DIR, exist_ok=True)
CHECKPOINT_PATH = os.path.join(DRIVE_DIR, "mtt_checkpoint.pt")
print(f"Checkpoint path: {CHECKPOINT_PATH}")


## §4 · Upload model.py and train.py
Upload the two source files from your local machine.  
Click the file picker that appears and select **both files at once**.

In [ ]:
from google.colab import files
import shutil, os

print("Select model.py and train.py when the file picker opens...")
uploaded = files.upload()   # opens the browser file picker

expected = {"model.py", "train.py"}
got = set(uploaded.keys())
missing = expected - got
extra   = got - expected

if missing:
    raise FileNotFoundError(
        f"Missing file(s): {missing}\n"
        "Please re-run this cell and upload BOTH model.py and train.py."
    )
if extra:
    print(f"⚠️  Unexpected file(s) ignored: {extra}")

# Move uploaded files to /content/ (upload() places them in cwd)
for fname in expected:
    src = os.path.join(os.getcwd(), fname)
    dst = f"/content/{fname}"
    shutil.move(src, dst)
    size = os.path.getsize(dst) / 1024
    print(f"✅  {fname:<12} → {dst}  ({size:.1f} KB)")

print("\nBoth files ready. Proceed to §5.")


## §5 · Configuration
Edit this cell to tune for your GPU or experiment settings.  
`CHECKPOINT_PATH` is set in §3; if you skipped Drive, it saves to `/content/`.

In [ ]:
import os, sys, torch
sys.path.insert(0, "/content")

# ─── GPU PRESETS ──────────────────────────────────────────────────────────────
# T4  (15 GB, free tier):  batch_size=4,  accum_steps=16  → eff. batch = 64
# L4  (22 GB, Pro):        batch_size=8,  accum_steps=8   → eff. batch = 64
# A100(40 GB, Pro+):       batch_size=16, accum_steps=4   → eff. batch = 64
# ─────────────────────────────────────────────────────────────────────────────

# If you skipped §3 Drive mount, checkpoint saves to /content/
try:
    _cp = CHECKPOINT_PATH
except NameError:
    _cp = "/content/mtt_checkpoint.pt"

CFG_OVERRIDES = dict(
    # ── data ──────────────────────────────────────────────────
    languages            = ["de", "fr", "vi", "es"],
    max_samples_per_pair = 50_000,

    # ── training schedule ─────────────────────────────────────
    train_steps          = 800,        # optimizer steps per cycle
    test_steps           = 200,        # eval steps per cycle

    # ── batch / memory (edit for your GPU) ────────────────────
    batch_size           = 4,          # T4 preset; increase for L4/A100
    accum_steps          = 16,         # keeps effective batch = 64

    # ── optimiser ─────────────────────────────────────────────
    lr                   = 5e-5,
    grad_clip            = 1.0,
    warmup_steps         = 400,
    lr_min               = 1e-7,

    # ── checkpoint ────────────────────────────────────────────
    checkpoint_path      = _cp,
    checkpoint_minutes   = 8.0,        # save often — Colab can disconnect

    # ── system ────────────────────────────────────────────────
    device               = "cuda" if torch.cuda.is_available() else "cpu",
    num_workers          = 2,          # Colab works best with 2
)
print("Config:")
for k, v in CFG_OVERRIDES.items():
    print(f"  {k:<26} = {v}")


## §6 · Keep-Alive (Optional)
Runs a JavaScript click loop to prevent Colab from timing out during long training.  
**Run this cell, then immediately run §7 and §8 in new cells.**

In [ ]:
from IPython.display import display, Javascript

display(Javascript('''
function clickConnect() {
  console.log("Keep-alive click at", new Date().toLocaleTimeString());
  try {
    document.querySelector("#top-toolbar > colab-connect-button")
      .shadowRoot.querySelector("#connect").click();
  } catch(e) {}
}
setInterval(clickConnect, 60000);
console.log("Keep-alive started — clicks every 60 s");
'''))
print("Keep-alive JS injected.")


## §7 · Load spaCy + Fetch OPUS-100 Data

In [ ]:
import sys, importlib
sys.path.insert(0, "/content")

import train as T
importlib.reload(T)          # pick up any edits to train.py

# Apply config overrides
T.CFG.update(CFG_OVERRIDES)

# Load spaCy NER model
nlp = T._load_spacy("xx_ent_wiki_sm")

# Fetch data (cached by HuggingFace datasets library after first download)
print("\nFetching OPUS-100 data — first run takes ~5–10 min...\n")
train_by_lang, test_by_lang = T.fetch_all_pairs()
print("\n✅ Data ready")


## §8 · Build Model + Train
Re-run this cell after a Colab disconnect — it automatically resumes from the  
last checkpoint saved to Drive (§3).

In [ ]:
import sys, gc, torch, importlib
sys.path.insert(0, "/content")

import model as M, train as T
importlib.reload(M)
importlib.reload(T)
T.CFG.update(CFG_OVERRIDES)

# Free any previously loaded model
try:
    del mtt
    gc.collect()
    torch.cuda.empty_cache()
except NameError:
    pass

# Initialise model (loads checkpoint automatically if it exists)
print("Initialising MTT model...")
mtt = M.MTT()
mtt.paramsCalc()

print("\nStarting training loop  (Ctrl+C or interrupt kernel to stop safely)\n")
try:
    T.train(mtt, train_by_lang, test_by_lang, nlp)
except KeyboardInterrupt:
    print("\n⏹ Training interrupted by user.")


## §9 · Inference — Test Translations

In [ ]:
# Run after training (or after loading a checkpoint below)
# 'mtt' must be defined (§8 must have run at least once this session)

device = CFG_OVERRIDES["device"]
mtt.eval()

examples = [
    ("en", "de", "The European Central Bank raised interest rates for the third time this year."),
    ("en", "fr", "Scientists discovered a new species of deep-sea fish near the Mariana Trench."),
    ("en", "vi", "The government announced a new infrastructure investment plan worth 10 billion dollars."),
    ("de", "en", "Die Konferenz wurde aufgrund des schlechten Wetters auf nächste Woche verschoben."),
    ("fr", "es", "Le président a signé un accord de coopération avec plusieurs pays africains."),
]

print(f"{'─'*70}")
print(f"  {'SRC LANG':<8}  {'TGT LANG':<8}  SOURCE / TRANSLATION")
print(f"{'─'*70}")
for src_lang, tgt_lang, text in examples:
    translations = mtt.translate(
        srcText=[text],
        targetLang=tgt_lang,
        maxNewTokens=128,
        numBeams=4,
        device=device,
    )
    print(f"  {src_lang} → {tgt_lang}  {text[:60]}")
    print(f"           ↳ {translations[0]}")
    print()


## §10 · Load Checkpoint (standalone)
Use this to load a saved checkpoint without re-running training.  
Useful after a session restart.

In [ ]:
import sys, torch
sys.path.insert(0, "/content")

import model as M

device = "cuda" if torch.cuda.is_available() else "cpu"

try:
    _cp = CHECKPOINT_PATH
except NameError:
    _cp = "/content/mtt_checkpoint.pt"

if not __import__("os").path.exists(_cp):
    print(f"No checkpoint found at {_cp}")
else:
    mtt = M.MTT.load(_cp, device=device)
    print(f"\nLoaded checkpoint from {_cp}")
    print("Ready for inference — run §9")


## §11 · Download Checkpoint to Local Machine

In [ ]:
from google.colab import files
import os

try:
    _cp = CHECKPOINT_PATH
except NameError:
    _cp = "/content/mtt_checkpoint.pt"

if os.path.exists(_cp):
    print(f"Downloading {_cp} ...")
    files.download(_cp)
else:
    print(f"No checkpoint found at {_cp} — train first (§8)")
